# Profiling and Cleaning: Saskatchewan Public-Sector Jobs

Data-quality review of the postings collected by the scraper. **This notebook contains no scraper code.**
It only reads the scraper's output (MongoDB `sk_jobs.postings`, or a master CSV) and writes review tables.

| Section | Output |
|---|---|
| 1. Data profile | rows per source/status, missing-value rates for every column |
| 2. City review | exact spelling variants + fuzzy pairs, for a person to confirm (nothing is overwritten) |
| 3. Data-quality flags | one row per flagged record + a summary per employer |
| 4. Job families | category → job family map built from the real category list, with coverage report |
| 5. Duplicates & reposts | parallel postings (same job, same day) and true reposts (closed, then re-listed) |
| 6. Save | CSVs in `OUTPUT_DIR`, and optionally collections in `sk_jobs_nlp` |

Runs locally (VS Code / Jupyter), on GitHub Actions, or in Colab. The MongoDB connection string comes from the
`MONGODB_URI` environment variable; if it isn't set you are prompted for it.

## 0. Settings

In [ ]:
DATA_SOURCE = "mongo"          # "mongo" (sk_jobs.postings) or "csv" (a master CSV file)
SOURCE_DB = "sk_jobs"          # use "sk_jobs_test" to check against test runs
CSV_PATH = "saskatchewan_public_sector_jobs_master.csv"   # only used when DATA_SOURCE = "csv"

OUTPUT_DIR = "profiling_output"
SAVE_TO_MONGO = True           # also write the review tables to the NLP database
NLP_DB = "sk_jobs_nlp"

CITY_SIMILARITY = 0.80         # fuzzy city pairs at or above this score are listed for review
SHORT_DESCRIPTION_CHARS = 50

## 1. Load

In [ ]:
import os
import re
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
TODAY = pd.Timestamp(datetime.now(ZoneInfo("America/Regina")).date())


def get_mongo_uri() -> str:
    uri = os.environ.get("MONGODB_URI")
    if not uri:
        from getpass import getpass
        uri = getpass("MongoDB connection string (MONGODB_URI): ").strip()
    if not uri:
        raise RuntimeError("No MongoDB connection string. Set the MONGODB_URI environment variable.")
    return uri


def mongo_client():
    try:
        from pymongo import MongoClient
    except ImportError as exc:
        raise ImportError("pymongo is missing: pip install pymongo") from exc
    return MongoClient(get_mongo_uri(), serverSelectionTimeoutMS=15000)


def load_postings() -> pd.DataFrame:
    if DATA_SOURCE == "mongo":
        client = mongo_client()
        docs = list(client[SOURCE_DB]["postings"].find({}, {"_id": 0}))
        if not docs:
            raise RuntimeError(f"{SOURCE_DB}.postings is empty")
        df = pd.DataFrame(docs)
        df["Posting_Status"] = df["status"].astype(str).str.title()
        origin = f"mongodb:{SOURCE_DB}.postings"
    elif DATA_SOURCE == "csv":
        if not Path(CSV_PATH).exists():
            raise FileNotFoundError(f"CSV not found: {CSV_PATH}")
        df = pd.read_csv(CSV_PATH)
        # A daily master only holds postings listed that day, so they are all open.
        df["Posting_Status"] = "Open"
        origin = CSV_PATH
        # Judge "closing date passed" against the day the CSV was collected, not today.
        if "Collection_Date" in df.columns:
            collected = to_date(df["Collection_Date"]).max()
            if pd.notna(collected):
                global TODAY
                TODAY = collected
    else:
        raise ValueError('DATA_SOURCE must be "mongo" or "csv"')

    required = ["Record_ID", "Source", "Job_Title", "City", "Posting_Date", "Closing_Date",
                "Salary_Min", "Salary_Max", "Salary_Unit", "Description", "Category"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing expected columns: {missing}")

    # Empty strings count as missing everywhere below.
    df = df.replace(r"^\s*$", np.nan, regex=True)
    print(f"Loaded {len(df):,} postings from {origin}")
    return df


def to_date(series: pd.Series) -> pd.Series:
    """ISO dates or timestamps -> date only (time and time zone are dropped)."""
    return pd.to_datetime(series.astype("string").str[:10], format="%Y-%m-%d", errors="coerce")


df = load_postings()
df["_Posting_Date"] = to_date(df["Posting_Date"])
df["_Closing_Date"] = to_date(df["Closing_Date"])
df["_Salary_Min"] = pd.to_numeric(df["Salary_Min"], errors="coerce")
df["_Salary_Max"] = pd.to_numeric(df["Salary_Max"], errors="coerce")

## 2. Data profile
Row counts per employer and status, then the share of missing values for **every** column per employer.
Columns a source never fills (100% missing) usually mean the site doesn't publish that field, not a scraper bug;
a jump from one run to the next is what to watch.

In [ ]:
helper_cols = [c for c in df.columns if c.startswith("_")]
data_cols = [c for c in df.columns if c not in helper_cols]

counts = (df.pivot_table(index="Source", columns="Posting_Status", values="Record_ID",
                         aggfunc="count", fill_value=0)
            .assign(Total=lambda t: t.sum(axis=1))
            .sort_values("Total", ascending=False))
print("Postings per employer:")
display(counts)

missing_by_source = (df[data_cols].drop(columns="Source").isna()
                     .groupby(df["Source"]).mean().mul(100).round(1).T)
missing_by_source["All"] = df[data_cols].isna().mean().mul(100).round(1)
missing_by_source = missing_by_source.sort_values("All", ascending=False)
print("\nMissing values per column and employer (%):")
display(missing_by_source)

dup_ids = df["Record_ID"].duplicated().sum()
print(f"\nDuplicate Record_IDs: {dup_ids}" + ("  <-- should be 0" if dup_ids else ""))

## 3. City review
Two lists, both for a person to confirm. **Nothing in the data is changed.**

- **Spelling variants**: names that are identical once case and spaces are normalised (`REGINA`, `Regina `). These are near-certain duplicates.
- **Fuzzy pairs**: different names that look alike. Pairs where one name is contained in the other
  (e.g. *North Battleford* / *Battleford*) are marked *likely different places*.

In [ ]:
def city_key(name) -> str:
    return re.sub(r"\s+", " ", str(name)).strip().casefold()


cities = df.loc[df["City"].notna(), ["City"]].copy()
cities["City_Key"] = cities["City"].map(city_key)
spellings = cities.groupby(["City_Key", "City"]).size().rename("Count").reset_index()

variant_keys = spellings.groupby("City_Key")["City"].nunique().loc[lambda s: s > 1].index
city_variants = (spellings[spellings["City_Key"].isin(variant_keys)]
                 .sort_values(["City_Key", "Count"], ascending=[True, False]))
# Suggest the most common spelling (ties: prefer not ALL-CAPS), with spaces tidied.
city_variants["_caps"] = city_variants["City"].str.strip().str.isupper()
city_variants = city_variants.sort_values(["City_Key", "Count", "_caps"], ascending=[True, False, True])
city_variants["Suggested_Name"] = (city_variants.groupby("City_Key")["City"].transform("first")
                                   .str.replace(r"\s+", " ", regex=True).str.strip())
city_variants["City_As_Written"] = city_variants["City"].map(repr)   # shows stray spaces
city_variants = city_variants[["City_Key", "City_As_Written", "Count", "Suggested_Name"]]

key_counts = cities["City_Key"].value_counts()
keys = sorted(key_counts.index)
pair_rows = []
for i, a in enumerate(keys):
    for b in keys[i + 1:]:
        score = SequenceMatcher(None, a, b).ratio()
        contained = re.search(rf"\b{re.escape(a)}\b", b) or re.search(rf"\b{re.escape(b)}\b", a)
        # Contained names (Battleford / North Battleford) are always listed, whatever the score.
        if score >= CITY_SIMILARITY or contained:
            pair_rows.append({
                "City_A": a, "Count_A": key_counts[a],
                "City_B": b, "Count_B": key_counts[b],
                "Similarity": round(score, 2),
                "Note": "likely different places" if contained else "check",
            })
city_pairs = pd.DataFrame(pair_rows, columns=["City_A", "Count_A", "City_B", "Count_B", "Similarity", "Note"])
city_pairs = city_pairs.sort_values("Similarity", ascending=False, ignore_index=True)

print(f"Spelling variants: {city_variants['City_Key'].nunique()} city name(s) written more than one way")
display(city_variants if len(city_variants) else "None found")
print(f"\nLook-alike pairs (similarity >= {CITY_SIMILARITY}, or one name inside the other): {len(city_pairs)}")
display(city_pairs if len(city_pairs) else "None found")

## 4. Data-quality flags
Each rule has an ID and checks one thing, so a record is never counted twice for the same problem.

| Rule | Flag |
|---|---|
| Q1 | Closing date before posting date |
| Q2 | `Salary_Min` greater than `Salary_Max` |
| Q3 | `Salary_Unit` is *Unspecified* (scraper couldn't tell hourly from annual) |
| Q4 | Salary out of range for its unit (hourly $10–$200, monthly $1,000–$50,000, annual $15,000–$500,000) |
| Q5 | Closing date has passed but the posting is still **Open** |
| Q6 | Description missing |
| Q7 | Description present but very short (likely a broken scrape) |

Q4 covers inferred units too (*Hourly (inferred)*, *Annual (inferred)*). Units it doesn't recognise are skipped and listed below.

In [ ]:
unit = df["Salary_Unit"].astype("string").str.strip().str.lower()
lo, hi = df["_Salary_Min"], df["_Salary_Max"]
desc_len = df["Description"].astype("string").str.strip().str.len()

SALARY_RANGES = {"hourly": (10, 200), "monthly": (1_000, 50_000), "annual": (15_000, 500_000)}
unit_base = unit.str.extract(r"^(hourly|monthly|annual)", expand=False)
out_of_range = pd.Series(False, index=df.index)
for base, (low_ok, high_ok) in SALARY_RANGES.items():
    is_unit = unit_base.eq(base).fillna(False)
    bad = (lo.notna() & ((lo < low_ok) | (lo > high_ok))) | (hi.notna() & ((hi < low_ok) | (hi > high_ok)))
    out_of_range |= is_unit & bad

RULES = {
    "Q1 Closing date before posting date":
        df["_Closing_Date"].notna() & df["_Posting_Date"].notna() & (df["_Closing_Date"] < df["_Posting_Date"]),
    "Q2 Salary_Min greater than Salary_Max":
        lo.notna() & hi.notna() & (lo > hi),
    "Q3 Salary unit unspecified":
        unit.eq("unspecified").fillna(False),
    "Q4 Salary out of range for its unit":
        out_of_range,
    "Q5 Closing date passed on Open posting":
        df["Posting_Status"].eq("Open") & df["_Closing_Date"].notna() & (df["_Closing_Date"] < TODAY),
    "Q6 Description missing":
        desc_len.isna() | desc_len.eq(0),
    "Q7 Description very short":
        desc_len.gt(0).fillna(False) & desc_len.lt(SHORT_DESCRIPTION_CHARS).fillna(False),
}

flag_frames = []
for rule, mask in RULES.items():
    mask = mask.fillna(False).astype(bool)
    if mask.any():
        hit = df.loc[mask, ["Record_ID", "Source", "Job_Title", "Posting_Date", "Closing_Date",
                            "Salary_Min", "Salary_Max", "Salary_Unit"]].copy()
        hit.insert(0, "Rule", rule)
        flag_frames.append(hit)
quality_flags = (pd.concat(flag_frames, ignore_index=True) if flag_frames
                 else pd.DataFrame(columns=["Rule", "Record_ID", "Source", "Job_Title"]))

rule_summary = (pd.DataFrame({rule: mask.fillna(False).astype(bool).groupby(df["Source"]).sum()
                              for rule, mask in RULES.items()}).T)
rule_summary["All"] = rule_summary.sum(axis=1)
print(f"{quality_flags['Record_ID'].nunique():,} of {len(df):,} postings have at least one flag")
print("\nFlags per rule and employer:")
display(rule_summary)

if len(quality_flags):
    examples = (quality_flags.groupby(["Rule", "Source"])["Record_ID"]
                .agg(Count="count", Examples=lambda s: ", ".join(s.astype(str).head(3)))
                .reset_index())
    display(examples)

other_units = unit[unit.notna() & unit_base.isna() & unit.ne("unspecified")].value_counts()
if len(other_units):
    print("Units not range-checked by Q4:")
    display(other_units)

## 5. Job families
Every employer labels categories differently. Instead of a short hand-written list that only matches a few exact values,
categories are mapped by **keyword rules** (first match wins), and every category the rules miss is listed with its count
so the rules can be extended. The raw `Category` column is never changed.

To fix a category the rules get wrong, add it to `CATEGORY_OVERRIDES` (exact category text → family).

In [ ]:
JOB_FAMILY_RULES = [  # (family, regex matched against the category, case-insensitive)
    ("Healthcare", r"nurs|health|medical|clinic|care aide|continuing care|pharm|therap|lab(oratory)?\b|diagnos|imaging|mental|patient|physician|dietary|dental"),
    ("IT & Data", r"information tech|\bit\b|technolog|software|digital|data|cyber|systems|network|telecom"),
    ("Engineering", r"engineer|technical|technolog(ist|y services)|design|drafting|scien"),
    ("Trades & Operations", r"trade|electric|mechanic|operator|operations|maintenance|construction|apprentic|journeyperson|power line|plant|field|equipment|facilit|labour"),
    ("Finance & Accounting", r"financ|account|audit|budget|payroll|tax|actuar|revenue|economic"),
    ("Customer Service & Sales", r"customer|sales|retail|call cent|contact cent|client serv|store|marketing"),
    ("Education & Social Services", r"educat|teach|learning|social|child|family|community|youth|correction|justice|probation|counsel"),
    ("Policy, Legal & Government", r"policy|legal|law|regulat|compliance|government|program|legislat|research|planning|analyst"),
    ("HR & Communications", r"human res|\bhr\b|talent|recruit|communicat|public affairs|media"),
    ("Administration", r"admin|clerical|office|reception|secretar|records|support services"),
    ("Management & Executive", r"manage|director|executive|leadership|supervis"),
    ("Students & Talent Pools", r"student|intern|co-?op|summer|talent pool|future opportunit"),
]
CATEGORY_OVERRIDES = {
    # "Exact category text": "Family",
}


def job_family(category) -> str:
    if pd.isna(category):
        return "No Category"
    text = str(category).strip()
    if text in CATEGORY_OVERRIDES:
        return CATEGORY_OVERRIDES[text]
    for family, pattern in JOB_FAMILY_RULES:
        if re.search(pattern, text, re.I):
            return family
    return "Unmapped"


df["Job_Family"] = df["Category"].map(job_family)

job_family_map = (df.groupby(["Source", "Category"], dropna=False).size().rename("Postings")
                    .reset_index())
job_family_map["Job_Family"] = job_family_map["Category"].map(job_family)
job_family_map = job_family_map.sort_values(["Job_Family", "Postings"], ascending=[True, False],
                                            ignore_index=True)

coverage = (df.assign(Mapped=~df["Job_Family"].isin(["Unmapped", "No Category"]))
              .groupby("Source")
              .agg(Postings=("Record_ID", "count"),
                   No_Category_pct=("Job_Family", lambda s: round(s.eq("No Category").mean() * 100, 1)),
                   Mapped_pct=("Mapped", lambda s: round(s.mean() * 100, 1))))
print("Job-family coverage per employer:")
display(coverage)

print("\nPostings per job family:")
display(df["Job_Family"].value_counts().to_frame("Postings"))

unmapped = job_family_map[job_family_map["Job_Family"] == "Unmapped"]
print(f"\nUnmapped categories ({len(unmapped)}): extend JOB_FAMILY_RULES or CATEGORY_OVERRIDES")
display(unmapped.head(40) if len(unmapped) else "None")

## 6. Duplicates and reposts
Two different things, kept apart:

- **Parallel postings**: open postings with the same title, organisation and city but different `Record_ID`s.
  Often legitimate (several identical SHA positions in one city), so this is a count to look at, not an error.
- **Reposts**: a posting that **closed** and was later **listed again under a new `Record_ID`**.
  This needs posting history (`first_seen` / `closed_date`), so it only runs when reading from MongoDB.

Rows with a missing title, organisation or city are left out, so missing values are never matched to each other.

In [ ]:
def norm(series: pd.Series) -> pd.Series:
    return series.astype("string").str.replace(r"\s+", " ", regex=True).str.strip().str.casefold()


org_col = "Organization" if "Organization" in df.columns else "Source"
df["_Match_Key"] = norm(df["Job_Title"]) + " | " + norm(df[org_col]) + " | " + norm(df["City"])

open_posts = df[(df["Posting_Status"] == "Open") & df["_Match_Key"].notna()]
group_size = open_posts.groupby("_Match_Key")["Record_ID"].transform("nunique")
parallel_postings = (open_posts[group_size > 1]
                     .assign(Group_Size=group_size[group_size > 1])
                     [["Group_Size", "Source", "Job_Title", org_col, "City", "Record_ID", "Posting_Date"]]
                     .sort_values(["Group_Size", "Job_Title"], ascending=[False, True], ignore_index=True))
print(f"Parallel postings: {len(parallel_postings):,} open postings share a title/organisation/city "
      f"({open_posts[group_size > 1]['_Match_Key'].nunique():,} groups)")
display(parallel_postings.head(15))

reposts = pd.DataFrame()
if {"first_seen", "closed_date"}.issubset(df.columns):
    hist = df[df["_Match_Key"].notna()].copy()
    hist["_first_seen"] = to_date(hist["first_seen"])
    hist["_closed_date"] = to_date(hist["closed_date"])
    closed = hist[hist["Posting_Status"] == "Closed"][["_Match_Key", "Record_ID", "_closed_date"]]
    later = hist[["_Match_Key", "Record_ID", "_first_seen", "Source", "Job_Title", org_col, "City", "Posting_Status"]]
    pairs = closed.merge(later, on="_Match_Key", suffixes=("_Closed", "_New"))
    pairs = pairs[(pairs["Record_ID_Closed"] != pairs["Record_ID_New"])
                  & (pairs["_first_seen"] >= pairs["_closed_date"])]
    # keep the first re-listing after each closure
    pairs = pairs.sort_values("_first_seen").drop_duplicates("Record_ID_Closed")
    reposts = (pairs.assign(Days_Until_Relisted=(pairs["_first_seen"] - pairs["_closed_date"]).dt.days,
                            Closed_On=pairs["_closed_date"].dt.date,
                            Relisted_On=pairs["_first_seen"].dt.date)
               [["Source", "Job_Title", org_col, "City", "Record_ID_Closed", "Closed_On",
                 "Record_ID_New", "Relisted_On", "Days_Until_Relisted", "Posting_Status"]]
               .rename(columns={"Posting_Status": "New_Status"})
               .sort_values("Relisted_On", ascending=False, ignore_index=True))
    print(f"\nReposts: {len(reposts):,} closed postings were listed again under a new Record_ID")
    display(reposts.head(15) if len(reposts) else "None found")
else:
    print("\nReposts: skipped (needs first_seen / closed_date history from MongoDB)")

## 7. Save
CSVs go to `OUTPUT_DIR`. With `SAVE_TO_MONGO = True` the same tables replace the matching collections in the NLP
database, each document tagged with the run date.

In [ ]:
outputs = {
    "data_profile_missing": missing_by_source.reset_index(names="Column"),
    "city_variants": city_variants,
    "city_pairs": city_pairs,
    "quality_flags": quality_flags,
    "quality_flag_summary": rule_summary.reset_index(names="Rule"),
    "job_family_map": job_family_map,
    "parallel_postings": parallel_postings,
    "reposts": reposts,
    "posting_job_family": df[["Record_ID", "Source", "Category", "Job_Family"]],
}

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
for name, table in outputs.items():
    table.to_csv(out_dir / f"{name}.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(outputs)} tables to {out_dir.resolve()}")


def to_documents(table: pd.DataFrame, run_date: str) -> list[dict]:
    clean = table.astype(object).where(table.notna(), None)
    docs = clean.to_dict("records")
    for d in docs:
        for k, v in d.items():
            if hasattr(v, "item"):
                d[k] = v.item()
            elif isinstance(v, (pd.Timestamp, datetime)):
                d[k] = v.isoformat()
            elif v is not None and not isinstance(v, (str, int, float, bool, list, dict)):
                d[k] = str(v)
        d["Profile_Run_Date"] = run_date
    return docs


if SAVE_TO_MONGO:
    db = mongo_client()[NLP_DB]
    run_date = TODAY.date().isoformat()
    for name, table in outputs.items():
        coll = db[f"profile_{name}"]
        coll.delete_many({})
        if len(table):
            coll.insert_many(to_documents(table, run_date))
        print(f"{NLP_DB}.profile_{name}: {len(table):,} documents")